In [2]:
free -h 
nvidia-smi
# 查看逻辑 CPU 总数
nproc

# 或者查看详细的架构信息
lscpu | grep -E '^CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket'

SyntaxError: invalid syntax (1370132569.py, line 7)

In [ ]:
aistudio@jupyter-2563417-10053501:~$ nproc
64
aistudio@jupyter-2563417-10053501:~$ # 或者查看详细的架构信息
aistudio@jupyter-2563417-10053501:~$ lscpu | grep -E '^CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket'
CPU(s):                             64
Thread(s) per core:                 2
Core(s) per socket:                 16
aistudio@jupyter-2563417-10053501:~$ top -bn1 | grep "colmap"
 247319 aistudio  20   0   13.4g 375008  30120 S 231.2   0.1  23:42.15 colmap
aistudio@jupyter-2563417-10053501:~$ 

In [ ]:
# for conda install  is too slow to install boost,use mamba instead  
source activate MyEnvForSD
conda install -c conda-forge mamba  
mamba --version
mamba install -c conda-forge boost boost-cpp eigen ceres-solver

In [ ]:
 
 source activate MyEnvForSD
 conda install -c conda-forge colmap

In [ ]:
# need to install cudaversion colmap pycolmap

In [ ]:
pip install pycolmap-cuda12 --index-url https://pypi.org/simple

In [ ]:
pip install pycolmap-cuda12 --index-url https://pypi.org/simple

In [ ]:
git clone https://github.com/graphdeco-inria/gaussian-splatting.git
cd gaussian-splatting
git submodule update --init --recursive --depth 1
cd gaussian-splatting
sorce activate MyEnvForSD
pip install /home/aistudio/models/gaussian-splatting/submodules/simple-knn --no-build-isolation
pip install /home/aistudio/models/gaussian-splatting/submodules/diff-gaussian-rasterization --no-build-isolation

# compile colmap gpu version

In [ ]:
cd ~/models
git clone --depth 1 https://github.com/colmap/colmap.git
cd colmap

git submodule update --init --recursive --depth 1


conda install -c conda-forge boost boost-cpp eigen ceres-solver glew freeglut qt




In [ ]:
# for conda install  is too slow to install boost,use mamba instead  
conda install -c conda-forge mamba  
mamba --version
mamba install -c conda-forge boost boost-cpp eigen ceres-solver

#cp -r .conda/envs/MyEnvForSD/ ./work/MyEnvFor3DColmapMamba/




# test

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("cuDNN version:", torch.backends.cudnn.version())
    print("Number of GPUs:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8


RuntimeError: cuDNN version incompatibility: PyTorch was compiled  against (9, 10, 2) but found runtime version (9, 5, 1). PyTorch already comes bundled with cuDNN. One option to resolving this error is to ensure PyTorch can find the bundled cuDNN. Looks like your LD_LIBRARY_PATH contains incompatible version of cudnn. Please either remove it from the path or install cudnn (9, 10, 2)

In [1]:
import torch
import diff_gaussian_rasterization
import simple_knn

print('✅ Successfully imported 3DGS submodules')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('❌ CUDA is not detected by PyTorch.')

✅ Successfully imported 3DGS submodules
CUDA available: True
GPU: Tesla V100-SXM2-32GB


In [ ]:
# Move the .bin files from sparse/0 to distorted/sparse/0
mv /home/aistudio/models/ColmapOutputs/Watt/sparse/0/*.bin /home/aistudio/models/ColmapOutputs/Watt/distorted/sparse/0/
# Create a symbolic link named 'input' inside the Watt folder
ln -s /home/aistudio/models/realityscan/Watt_Resized /home/aistudio/models/ColmapOutputs/Watt/input

In [ ]:
 pip install plyfile
 

![Colmap output folder ](colmapout_folder_hier.jpg)



# create colmap folder strucutres

In [ ]:
# 1. 创建核心目录
export PROJECT_PATH=/home/aistudio/models/realityscan/MyBicycles
mkdir -p $PROJECT_PATH/input
mkdir -p $PROJECT_PATH/sparse

# 2. 检查路径
echo "请确保你的 100 张照片已存放在: $PROJECT_PATH/input/images"

In [ ]:

#cpu version for feature extraction and matching
# 特征提取

# need to set use_gpu 0  for cpu,cannot use gpu for some reason
colmap feature_extractor \
    --database_path $PROJECT_PATH/dist/database.db \
    --image_path $PROJECT_PATH/input/images \
    --ImageReader.single_camera 0 \
    --ImageReader.camera_model PINHOLE \
    --SiftExtraction.use_gpu 0 \
    --SiftExtraction.num_threads 64

# 穷举匹配 (100张图大约需要 100 分钟)
colmap exhaustive_matcher \
    --database_path $PROJECT_PATH/dist/database.db \
    --SiftMatching.use_gpu 0 \
    --SiftMatching.num_threads 64

# 稀疏重建

colmap mapper \
    --database_path $PROJECT_PATH/dist/database.db \
    --image_path $PROJECT_PATH/input/images \
    --output_path $PROJECT_PATH/dist/sparse \
    --Mapper.num_threads 64 \
    --Mapper.ba_global_images_ratio 1.1 \
    --Mapper.ba_global_points_ratio 1.1 \
    --Mapper.ba_global_max_num_iterations 50 \
    --Mapper.init_min_num_inliers 50

In [ ]:
colmap image_undistorter \
    --image_path $PROJECT_PATH/input/images \
    --input_path $PROJECT_PATH/dist/sparse/0 \
    --output_path $PROJECT_PATH/dist/undistorted \
    --output_type COLMAP

In [ ]:
python /home/aistudio/models/gaussian-splatting/train.py \
    -s /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted \
    --model_path /home/aistudio/models/ColmapOutputs/MyBicycles/output \
    --save_iterations 7000 14000 21000 28000 30000 \
    --iterations 30000

In [ ]:
#gpu version of colmap
# 特征提取

# need to set use_gpu 0  for cpu,cannot use gpu for some reason
colmap feature_extractor \
    --database_path $PROJECT_PATH/dist/database.db \
    --image_path $PROJECT_PATH/input/images \
    --ImageReader.single_camera 0 \
    --ImageReader.camera_model PINHOLE \
    --SiftExtraction.use_gpu 1 \
    --SiftExtraction.num_threads 64

# 穷举匹配 (100张图大约需要 100 分钟)  gpu only 1 minutes
colmap exhaustive_matcher \
    --database_path $PROJECT_PATH/dist/database.db \
    --SiftMatching.use_gpu 1 \
    --SiftMatching.num_threads 64

# 稀疏重建
# gpu for 8 minutes,cpu for 2hours
colmap mapper \
    --database_path $PROJECT_PATH/dist/database.db \
    --image_path $PROJECT_PATH/input/images \
    --output_path $PROJECT_PATH/dist/sparse \
    --Mapper.num_threads 64 \
    --Mapper.ba_global_images_ratio 1.1 \
    --Mapper.ba_global_points_ratio 1.1 \
    --Mapper.ba_global_max_num_iterations 50 \
    --Mapper.init_min_num_inliers 50

In [ ]:
export PROJECT_PATH=/home/aistudio/models/CherryCampus/Campus1024
# 1. 创建核心目录
mkdir -p $PROJECT_PATH/input/images
mkdir -p $PROJECT_PATH/sparse

# 2. 检查路径
echo "请确保你的 100 张照片已存放在: $PROJECT_PATH/input/images"

In [ ]:
colmap feature_extractor \
    --database_path $PROJECT_PATH/database.db \
    --image_path $PROJECT_PATH/input/images \
    --ImageReader.single_camera 0 \
    --ImageReader.camera_model PINHOLE \
    --SiftExtraction.use_gpu 1 \
    --SiftExtraction.num_threads 64

# 穷举匹配 (100张图大约需要 100 分钟)  gpu only 1 minutes
colmap exhaustive_matcher \
    --database_path $PROJECT_PATH/database.db \
    --SiftMatching.use_gpu 1 \
    --SiftMatching.num_threads 64
# for 2k images up use sequential matcher for simplicity
colmap sequential_matcher \
    --database_path $PROJECT_PATH/database.db \
    --SiftMatching.use_gpu 1 \
    --SequentialMatching.overlap 20

# 稀疏重建
# gpu for 8 minutes,cpu for 2hours
colmap mapper \
    --database_path $PROJECT_PATH/database.db \
    --image_path $PROJECT_PATH/input/images \
    --output_path $PROJECT_PATH/sparse \
    --Mapper.num_threads 64 \
    --Mapper.init_min_num_inliers 100 \
    --Mapper.ba_global_max_num_iterations 100

colmap image_undistorter \
    --image_path $PROJECT_PATH/input/images \
    --input_path $PROJECT_PATH/sparse/0 \
    --output_path $PROJECT_PATH/dense/undistorted \
    --output_type COLMAP

In [ ]:
python /home/aistudio/models/gaussian-splatting/train.py \
    -s /home/aistudio/models/CherryCampus/Campus1024/dense/undistorted \
    --model_path /home/aistudio/models/CherryCampus/Campus1024/output \
    --save_iterations 7000 14000 21000 28000 30000 \
    --iterations 30000

In [ ]:
import pycolmap
from pathlib import Path

# 定义路径
project_path = Path("/home/aistudio/models/realityscan/MyBicycles")
database_path = project_path / "dist/database.db"
image_path = project_path / "input/images"

# 1. 特征提取 (可以使用 GPU)
# PyCOLMAP 内部处理了无头环境的上下文，通常能直接调起 CUDA
pycolmap.extract_features(
    database_path=database_path,
    image_path=image_path,
    camera_model="PINHOLE",
    # 如果 GPU 依然不可用，PyCOLMAP 会自动退回到 CPU，
    # 或者你可以手动指定 SiftExtractionOptions
)

# 2. 穷举匹配
pycolmap.match_exhaustive(database_path=database_path)

# 3. 增量式重建 (对应 colmap mapper)
output_path = project_path / "dist/sparse"
output_path.mkdir(parents=True, exist_ok=True)

reconstructions = pycolmap.incremental_mapping(
    database_path=database_path,
    image_path=image_path,
    output_path=output_path
)

print(f"重建完成！共生成 {len(reconstructions)} 个模型。")

In [ ]:
python train.py -s /home/aistudio/models/ColmapOutputs/Watt --model_path /home/aistudio/models/ColmapOutputs/Watt/output

In [ ]:
colmap model_analyzer --path /home/aistudio/models/ColmapOutputs/MyBicycles/sparse/0 

In [ ]:
# mothod1 works fine need add sparse/0
# use image undistorter to undistort images  gen  undistorted folder

colmap image_undistorter \
    --image_path /home/aistudio/models/ColmapOutputs/MyBicycles/images \
    --input_path /home/aistudio/models/ColmapOutputs/MyBicycles/sparse/0 \
    --output_path /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted \
    --output_type COLMAP 
#colmap output_path  /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/ 
# but 3dgs train need to create a folder 0 and move the file to the subfolder 0

mkdir -p /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/0

mv /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/*.bin \
   /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/0/

# mothod2 works fine
# the output folder 
#python /home/aistudio/models/gaussian-splatting/convert.py \
#    -s /home/aistudio/models/ColmapOutputs/MyBicycles
    
python /home/aistudio/models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted \
-m /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/output \
--save_iterations 7000 14000 21000 28000 30000
#  first will convert point3d.bin to point3d.ply
# then train the model in 30000 steps every 7000 steps save a model
 

In [ ]:
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/gaussian-splatting$ pip install plyfile
Looking in indexes: http://mirrors.baidubce.com/pypi/simple/
Collecting plyfile
  Downloading http://mirrors.baidubce.com/pypi/packages/22/22/1755bb4c7db15bb1ed63b4eb7a7fc133bf42a3f9cc806c0d5941e107ba90/plyfile-1.1.3-py3-none-any.whl (36 kB)
Requirement already satisfied: numpy>=1.21 in /home/aistudio/external-libraries/lib/python3.10/site-packages (from plyfile) (2.2.6)
Installing collected packages: plyfile
Successfully installed plyfile-1.1.3
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/gaussian-splatting$ python train.py -s /home/aistudio/models/ColmapOutputs/Watt --model_path /home/aistudio/models/ColmapOutputs/Watt/output
Optimizing /home/aistudio/models/ColmapOutputs/Watt/output
Output folder: /home/aistudio/models/ColmapOutputs/Watt/output [29/03 18:46:52]
Tensorboard not available: not logging progress [29/03 18:46:52]
Reading camera 415/415 [29/03 18:46:54]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [29/03 18:46:54]
Loading Training Cameras [29/03 18:46:55]
Loading Test Cameras [29/03 18:47:35]
Number of points at initialisation :  108160 [29/03 18:47:35]
Training progress:  23%|█████████████▎                                           | 7000/30000 [05:08<21:14, 18.05it/s, Loss=0.1032912, Depth Loss=0.0000000]
[ITER 7000] Evaluating train: L1 0.05458241552114487 PSNR 21.37511329650879 [29/03 18:52:45]

[ITER 7000] Saving Gaussians [29/03 18:52:45]
Training progress: 100%|████████████████████████████████████████████████████████| 30000/30000 [28:52<00:00, 17.31it/s, Loss=0.0770907, Depth Loss=0.0000000]

[ITER 30000] Evaluating train: L1 0.044078940898180013 PSNR 22.83071517944336 [29/03 19:16:29]

[ITER 30000] Saving Gaussians [29/03 19:16:29]

Training complete. [29/03 19:16:52]

# use superspl.at to edit upload 

In [ ]:
https://superspl.at/ 
https://superspl.at/scene/1f9f0dd2/edit

# MyBicycles training logs


In [ ]:
aistudio@jupyter-2563417-10053501:~$ source activate MyEnvForSD
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~$ cd /home/aistudio/models/ColmapOutputs/MyBicycles
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ ls - al
ls: cannot access '-': No such file or directory
ls: cannot access 'al': No such file or directory
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ ls -al
total 28
drwxr-xr-x 1 aistudio aistudio 4096 Apr  2 09:52 .
drwxr-xr-x 1 aistudio aistudio 4096 Apr  1 11:59 ..
drwxr-xr-x 2 aistudio aistudio 4096 Apr  1 11:59 .ipynb_checkpoints
drwxr-xr-x 2 aistudio aistudio 4096 Apr  2 09:52 images
drwxr-xr-x 3 aistudio aistudio 4096 Apr  2 09:51 sparse
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ python train.py -s MyBicyclesmodels/ColmapOutputs/MyBicycles
python: can't open file '/home/aistudio/models/ColmapOutputs/MyBicycles/train.py': [Errno 2] No such file or directory
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ python models/gaussian-splatting/train.py -s MyBicyclesmodels/ColmapOutputs/MyBicycles
python: can't open file '/home/aistudio/models/ColmapOutputs/MyBicycles/models/gaussian-splatting/train.py': [Errno 2] No such file or directory
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ python models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles
python: can't open file '/home/aistudio/models/ColmapOutputs/MyBicycles/models/gaussian-splatting/train.py': [Errno 2] No such file or directory
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$  python /home/aistudio/models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles
Optimizing 
Output folder: ./output/885b6d10-a [02/04 10:11:37]
Tensorboard not available: not logging progress [02/04 10:11:37]
Reading camera 1/100Traceback (most recent call last):
  File "/home/aistudio/models/gaussian-splatting/train.py", line 282, in <module>
    training(lp.extract(args), op.extract(args), pp.extract(args), args.test_iterations, args.save_iterations, args.checkpoint_iterations, args.start_checkpoint, args.debug_from)
  File "/home/aistudio/models/gaussian-splatting/train.py", line 51, in training
    scene = Scene(dataset, gaussians)
  File "/home/aistudio/models/gaussian-splatting/scene/__init__.py", line 44, in __init__
    scene_info = sceneLoadTypeCallbacks["Colmap"](args.source_path, args.images, args.depths, args.eval, args.train_test_exp)
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 194, in readColmapSceneInfo
    cam_infos_unsorted = readColmapCameras(
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 98, in readColmapCameras
    assert False, "Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!"
AssertionError: Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ colmap image_undistorter \
>     --image_path /home/aistudio/models/ColmapOutputs/MyBicycles/images \
>     --input_path /home/aistudio/models/ColmapOutputs/MyBicycles/sparse/0 \
>     --output_path /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted \
>     --output_type COLMAP

==============================================================================
Reading reconstruction
==============================================================================

 => Reconstruction with 100 images and 52934 points

==============================================================================
Image undistortion
==============================================================================

Undistorting image [1/100]
Undistorting image [2/100]
Undistorting image [3/100]
Undistorting image [4/100]
Undistorting image [5/100]
Undistorting image [6/100]
Undistorting image [7/100]
Undistorting image [8/100]
Undistorting image [9/100]
Undistorting image [10/100]
Undistorting image [11/100]
Undistorting image [12/100]
Undistorting image [13/100]
Undistorting image [14/100]
Undistorting image [15/100]
Undistorting image [16/100]
Undistorting image [17/100]
Undistorting image [18/100]
Undistorting image [19/100]
Undistorting image [20/100]
Undistorting image [21/100]
Undistorting image [22/100]
Undistorting image [23/100]
Undistorting image [24/100]
Undistorting image [25/100]
Undistorting image [26/100]
Undistorting image [27/100]
Undistorting image [28/100]
Undistorting image [29/100]
Undistorting image [30/100]
Undistorting image [31/100]
Undistorting image [32/100]
Undistorting image [33/100]
Undistorting image [34/100]
Undistorting image [35/100]
Undistorting image [36/100]
Undistorting image [37/100]
Undistorting image [38/100]
Undistorting image [39/100]
Undistorting image [40/100]
Undistorting image [41/100]
Undistorting image [42/100]
Undistorting image [43/100]
Undistorting image [44/100]
Undistorting image [45/100]
Undistorting image [46/100]
Undistorting image [47/100]
Undistorting image [48/100]
Undistorting image [49/100]
Undistorting image [50/100]
Undistorting image [51/100]
Undistorting image [52/100]
Undistorting image [53/100]
Undistorting image [54/100]
Undistorting image [55/100]
Undistorting image [56/100]
Undistorting image [57/100]
Undistorting image [58/100]
Undistorting image [59/100]
Undistorting image [60/100]
Undistorting image [61/100]
Undistorting image [62/100]
Undistorting image [63/100]
Undistorting image [64/100]
Undistorting image [65/100]
Undistorting image [66/100]
Undistorting image [67/100]
Undistorting image [68/100]
Undistorting image [69/100]
Undistorting image [70/100]
Undistorting image [71/100]
Undistorting image [72/100]
Undistorting image [73/100]
Undistorting image [74/100]
Undistorting image [75/100]
Undistorting image [76/100]
Undistorting image [77/100]
Undistorting image [78/100]
Undistorting image [79/100]
Undistorting image [80/100]
Undistorting image [81/100]
Undistorting image [82/100]
Undistorting image [83/100]
Undistorting image [84/100]
Undistorting image [85/100]
Undistorting image [86/100]
Undistorting image [87/100]
Undistorting image [88/100]
Undistorting image [89/100]
Undistorting image [90/100]
Undistorting image [91/100]
Undistorting image [92/100]
Undistorting image [93/100]
Undistorting image [94/100]
Undistorting image [95/100]
Undistorting image [96/100]
Undistorting image [97/100]
Undistorting image [98/100]
Undistorting image [99/100]
Undistorting image [100/100]
Writing reconstruction...
Writing configuration...
Writing scripts...
Elapsed time: 0.432 [minutes]
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$  python /home/aistudio/models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles
Optimizing 
Output folder: ./output/9c374f8e-d [02/04 10:14:23]
Tensorboard not available: not logging progress [02/04 10:14:23]
Reading camera 1/100Traceback (most recent call last):
  File "/home/aistudio/models/gaussian-splatting/train.py", line 282, in <module>
    training(lp.extract(args), op.extract(args), pp.extract(args), args.test_iterations, args.save_iterations, args.checkpoint_iterations, args.start_checkpoint, args.debug_from)
  File "/home/aistudio/models/gaussian-splatting/train.py", line 51, in training
    scene = Scene(dataset, gaussians)
  File "/home/aistudio/models/gaussian-splatting/scene/__init__.py", line 44, in __init__
    scene_info = sceneLoadTypeCallbacks["Colmap"](args.source_path, args.images, args.depths, args.eval, args.train_test_exp)
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 194, in readColmapSceneInfo
    cam_infos_unsorted = readColmapCameras(
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 98, in readColmapCameras
    assert False, "Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!"
AssertionError: Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$  python /home/aistudio/models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted
Optimizing 
Output folder: ./output/fbb6bb37-2 [02/04 10:15:57]
Tensorboard not available: not logging progress [02/04 10:15:57]
Traceback (most recent call last):
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 149, in readColmapSceneInfo
    cam_extrinsics = read_extrinsics_binary(cameras_extrinsic_file)
  File "/home/aistudio/models/gaussian-splatting/scene/colmap_loader.py", line 187, in read_extrinsics_binary
    with open(path_to_model_file, "rb") as fid:
FileNotFoundError: [Errno 2] No such file or directory: '/home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/0/images.bin'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/aistudio/models/gaussian-splatting/train.py", line 282, in <module>
    training(lp.extract(args), op.extract(args), pp.extract(args), args.test_iterations, args.save_iterations, args.checkpoint_iterations, args.start_checkpoint, args.debug_from)
  File "/home/aistudio/models/gaussian-splatting/train.py", line 51, in training
    scene = Scene(dataset, gaussians)
  File "/home/aistudio/models/gaussian-splatting/scene/__init__.py", line 44, in __init__
    scene_info = sceneLoadTypeCallbacks["Colmap"](args.source_path, args.images, args.depths, args.eval, args.train_test_exp)
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 154, in readColmapSceneInfo
    cam_extrinsics = read_extrinsics_text(cameras_extrinsic_file)
  File "/home/aistudio/models/gaussian-splatting/scene/colmap_loader.py", line 249, in read_extrinsics_text
    with open(path, "r") as fid:
FileNotFoundError: [Errno 2] No such file or directory: '/home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/0/images.txt'
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ python models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles/distorted
python: can't open file '/home/aistudio/models/ColmapOutputs/MyBicycles/models/gaussian-splatting/train.py': [Errno 2] No such file or directory
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$  python /home/aistudio/models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted
Optimizing 
Output folder: ./output/2fa449bd-b [02/04 10:17:47]
Tensorboard not available: not logging progress [02/04 10:17:47]
Reading camera 100/100 [02/04 10:17:47]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [02/04 10:17:47]
Loading Training Cameras [02/04 10:17:47]
Loading Test Cameras [02/04 10:18:47]
Number of points at initialisation :  52934 [02/04 10:18:48]
Training progress:  23%|███████████▏                                    | 7000/30000 [15:15<46:53,  8.17it/s, Loss=0.1162505, Depth Loss=0.0000000]
[ITER 7000] Evaluating train: L1 0.05141039192676544 PSNR 21.697405242919924 [02/04 10:34:06]

[ITER 7000] Saving Gaussians [02/04 10:34:06]
Training progress: 100%|█████████████████████████████████████████████| 30000/30000 [1:10:29<00:00,  7.09it/s, Loss=0.0923409, Depth Loss=0.0000000]^A

[ITER 30000] Evaluating train: L1 0.02892332747578621 PSNR 26.950869750976565 [02/04 11:29:20]

[ITER 30000] Saving Gaussians [02/04 11:29:20]
Killed
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ 

In [ ]:

Pose refinement report
----------------------
    Residuals : 650
   Parameters : 8
   Iterations : 20
         Time : 0.00806093 [s]
 Initial cost : 0.841373 [px]
   Final cost : 0.637206 [px]
  Termination : Convergence

  => Continued observations: 302
  => Added observations: 273

Bundle adjustment report
------------------------
    Residuals : 72256
   Parameters : 1574
   Iterations : 26
         Time : 1.12501 [s]
 Initial cost : 0.43287 [px]
   Final cost : 0.398303 [px]
  Termination : No convergence

  => Merged observations: 112
  => Completed observations: 62
  => Filtered observations: 158
  => Changed observations: 0.009190

Bundle adjustment report
------------------------
    Residuals : 77432
   Parameters : 1112
   Iterations : 4
         Time : 0.127388 [s]
 Initial cost : 0.438 [px]
   Final cost : 0.434775 [px]
  Termination : Convergence

  => Merged observations: 22
  => Completed observations: 20
  => Filtered observations: 7
  => Changed observations: 0.001266

==============================================================================
Retriangulation
==============================================================================

  => Completed observations: 298
  => Merged observations: 342
  => Retriangulated observations: 0

==============================================================================
Global bundle adjustment
==============================================================================

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  2.082694e+05    0.00e+00    7.30e+04   0.00e+00   0.00e+00  1.00e+04        0    3.10e-01    3.18e+00

CHOLMOD version 5.3.1, Feb 20, 2025: Symbolic Analysis: status: OK
  Architecture:
    sizeof(int):      4
    sizeof(int64_t):  8
    sizeof(void *):   8
    sizeof(double):   8
    sizeof(Int):      4 (CHOLMOD's basic integer)
    sizeof(SUITESPARSE_BLAS_INT): 4 (integer used in the BLAS)
  Results from most recent analysis:
    Cholesky flop count: 1.4836e+08
    Nonzeros in L:       3.0016e+05
  memory blocks in use:          11
  memory in use (MB):           0.0
  peak memory usage (MB):       1.2
  maxrank:    update/downdate rank:   8
  supernodal control: 1 40 (supernodal if flops/lnz >= 40)
  nmethods:   number of ordering methods to try: 1
    method 0: natural
        flop count: 1.4836e+08
        nnz(L):     3.0016e+05
  OK
   1  2.061983e+05    2.07e+03    3.19e+03   0.00e+00   1.00e+00  3.00e+04        1    4.70e+00    7.88e+00
   2  2.061970e+05    1.25e+00    1.37e+04   2.14e+01   5.24e-01  3.00e+04        1    2.92e+00    1.08e+01
   3  2.061955e+05    1.52e+00    8.15e+03   1.57e+01   7.89e-01  3.71e+04        1    2.88e+00    1.37e+01
   4  2.061949e+05    6.30e-01    6.94e+03   1.44e+01   6.75e-01  3.88e+04        1    2.92e+00    1.66e+01
   5  2.061944e+05    5.09e-01    4.10e+03   1.10e+01   8.32e-01  5.48e+04        1    2.98e+00    1.96e+01
   6  2.061941e+05    2.29e-01    3.60e+03   1.03e+01   7.38e-01  6.15e+04        1    2.90e+00    2.25e+01
   7  2.061940e+05    1.65e-01    1.83e+03   7.26e+00   8.98e-01  1.24e+05        1    3.10e+00    2.56e+01
   8  2.061939e+05    6.07e-02    1.63e+03   6.84e+00   7.97e-01  1.57e+05        1    2.90e+00    2.85e+01
   9  2.061939e+05    3.25e-02    4.45e+02   3.55e+00   1.00e+00  4.72e+05        1    2.92e+00    3.14e+01
  10  2.061939e+05    5.53e-03    1.71e+02   2.24e+00   1.05e+00  1.41e+06        1    2.88e+00    3.43e+01
  11  2.061939e+05    1.03e-03    1.44e+01   7.47e-01   1.12e+00  4.24e+06        1    2.89e+00    3.72e+01
  12  2.061939e+05    2.89e-04    1.53e+00   2.66e-01   1.05e+00  1.27e+07        1    2.90e+00    4.01e+01
  13  2.061939e+05    1.03e-04    7.89e-01   8.56e-02   1.01e+00  3.82e+07        1    3.10e+00    4.32e+01


Bundle adjustment report
------------------------
    Residuals : 978204
   Parameters : 224995
   Iterations : 14
         Time : 43.2579 [s]
 Initial cost : 0.461422 [px]
   Final cost : 0.459117 [px]
  Termination : Convergence

  => Completed observations: 72
  => Merged observations: 141
  => Filtered observations: 23
  => Changed observations: 0.000483
  => Filtered images: 0

==============================================================================
Finding good initial image pair
==============================================================================

  => No good initial image pair found.

Elapsed time: 105.401 [minutes]
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/realityscan$ 